In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv(find_dotenv(), override=True)

model = ChatOpenAI(
    model=os.getenv("KIMI_MODEL"),
    base_url=os.getenv("KIMI_BASE_URL").rstrip("/") + "/v1",
    api_key="not-needed",
    default_headers={
        "Modal-Key": os.getenv("MODEL_KEY"),
        "Modal-Secret": os.getenv("MODEL_SECRET"),
    },
)


# state
class LLMState(TypedDict):
    question: str
    answer: str


# node
def llm_qa(state: LLMState) -> dict:
    q = state["question"]
    prompt = f"Answer the following question: {q}"
    answer = model.invoke(prompt).content
    return {"answer": answer}


# graph
graph = StateGraph(LLMState)
graph.add_node("llm_qa", llm_qa)

# edges
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

workflow = graph.compile()

if __name__ == "__main__":
    initial_state = {"question": "who are the declared most currpot politician in the world?"}
    final_state = workflow.invoke(initial_state)
    print(final_state['answer'])


IndentationError: unexpected indent (3225403664.py, line 46)